# Cross-Framework Transfer Analysis

Tests whether probes trained on Crescendo generalise to ActorAttack and X-Teaming.
All representations use Design D (last position of prompt, `add_generation_prompt=True`).

## Key questions

| Question | Method |
|----------|--------|
| Does a harmful/benign probe trained on Crescendo transfer to other frameworks? | Train on Crescendo, test on AA/XT |
| Is the signal framework-agnostic or framework-specific? | Full transfer matrix (all → all) |
| Do all frameworks show the same turn-by-turn trajectory? | LAT score by turn per framework |
| Does compliance depth predict refusal/outcome cross-framework? | n_context_turns effect per framework |

## Interpretation guide

- **High transfer AUC**: the representational signal is about harm content, not Crescendo mechanics
- **Low transfer AUC**: each framework creates framework-specific representational patterns
- **Transfer AUC ≈ within-framework AUC**: the probe is learning something universal about how the model encodes harm

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

sns.set_theme(style="whitegrid", font_scale=1.1)
FIG_DIR = repo_root / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

LAYER_INDICES = [1, 16, 32]
LAYER_NAMES   = [f"Layer {l}" for l in LAYER_INDICES]
LAYER_COLORS  = ["#4e79a7", "#f28e2b", "#e15759"]

print("Imports OK")

## Load all datasets

In [ ]:
REPR_ROOT = repo_root / "data" / "representations"

DATASETS = {
    # (framework, split) → (harmful_folder, benign_folder)
    "crescendo": (
        "crescendo_harmful_10_turns_faithful",
        "crescendo_benign_10_turns_faithful",
    ),
    "actorattack": (
        "actorattack_harmful_faithful",
        "actorattack_benign_faithful",
    ),
    "xteaming": (
        "xteaming_harmful_faithful",
        "xteaming_benign_faithful",
    ),
}

# data[framework] = {"states": np.ndarray, "meta": pd.DataFrame}
data = {}

for fw, (harm_dir, ben_dir) in DATASETS.items():
    parts_s, parts_m = [], []
    for folder, split in [(harm_dir, "harmful"), (ben_dir, "benign")]:
        d = REPR_ROOT / folder
        if not (d / "hidden_states.npy").exists():
            print(f"MISSING: {folder} — run notebook 14 first")
            continue
        s = np.load(str(d / "hidden_states.npy"), mmap_mode="r")
        m = pd.read_parquet(d / "metadata.parquet")
        parts_s.append(s)
        parts_m.append(m)

    if len(parts_s) == 2:
        states = np.concatenate(parts_s, axis=0).astype(np.float32)
        meta   = pd.concat(parts_m, ignore_index=True)
        data[fw] = {"states": states, "meta": meta}
        print(f"{fw}: {states.shape}  "
              f"harmful={meta['label'].sum()}  benign={(meta['label']==0).sum()}")

frameworks = list(data.keys())
print(f"\nLoaded: {frameworks}")

## Probe helpers

In [ ]:
def get_layer(states: np.ndarray, layer_pos: int) -> np.ndarray:
    return np.asarray(states[:, layer_pos, :], dtype=np.float32)


def lr_cv(X: np.ndarray, y: np.ndarray, n_splits: int = 5, C: float = 0.1) -> float:
    """Stratified k-fold LR. Returns mean AUC."""
    if len(np.unique(y)) < 2 or len(y) < n_splits * 2:
        return float("nan")
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []
    for tr, te in skf.split(X, y):
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xte = sc.transform(X[te])
        clf = LogisticRegression(max_iter=1000, C=C)
        clf.fit(Xtr, y[tr])
        aucs.append(roc_auc_score(y[te], clf.predict_proba(Xte)[:, 1]))
    return float(np.mean(aucs))


def train_probe(X_train: np.ndarray, y_train: np.ndarray, C: float = 0.1):
    """Train a single LR probe. Returns (scaler, clf)."""
    sc  = StandardScaler()
    Xtr = sc.fit_transform(X_train)
    clf = LogisticRegression(max_iter=1000, C=C)
    clf.fit(Xtr, y_train)
    return sc, clf


def eval_probe(sc, clf, X_test: np.ndarray, y_test: np.ndarray) -> float:
    """Evaluate a trained probe on new data. Returns AUC."""
    if len(np.unique(y_test)) < 2:
        return float("nan")
    Xte   = sc.transform(X_test)
    proba = clf.predict_proba(Xte)[:, 1]
    return roc_auc_score(y_test, proba)


def train_lat(X_train: np.ndarray, y_train: np.ndarray):
    """Train LAT reading vector (mean-diff). Returns (scaler, direction)."""
    sc      = StandardScaler()
    X_sc    = sc.fit_transform(X_train)
    vec     = X_sc[y_train == 1].mean(0) - X_sc[y_train == 0].mean(0)
    vec    /= np.linalg.norm(vec) + 1e-8
    return sc, vec


def eval_lat(sc, vec, X_test: np.ndarray, y_test: np.ndarray) -> float:
    if len(np.unique(y_test)) < 2:
        return float("nan")
    scores = sc.transform(X_test) @ vec
    return roc_auc_score(y_test, scores)


print("Helpers defined.")

## Within-framework baselines

Train and evaluate on the same framework (k-fold CV).
This is the ceiling — best possible AUC if we had in-distribution training data.

In [ ]:
within_aucs = {}   # (fw, layer_name) -> auc

print("Within-framework AUC (harmful vs benign, 5-fold CV):")
print(f"{'Framework':<15}" + "".join(f"{ln:>12}" for ln in LAYER_NAMES))
print("-" * (15 + 12 * len(LAYER_NAMES)))

for fw in frameworks:
    states = data[fw]["states"]
    y      = data[fw]["meta"]["label"].values
    row_str = f"{fw:<15}"
    for li, lname in enumerate(LAYER_NAMES):
        X   = get_layer(states, li)
        auc = lr_cv(X, y)
        within_aucs[(fw, lname)] = auc
        row_str += f"{auc:>12.3f}"
    print(row_str)

## Cross-framework transfer matrix

Train probe on one framework, evaluate on another (no CV — single train/test split
across frameworks). Each row = train framework; each column = test framework.
Diagonal = within-framework (same as above for reference).

In [ ]:
# Use best layer for main transfer analysis; show all layers below
TRANSFER_LAYER_POS = 2   # Layer 32 — best from nb12/13

transfer_matrix = pd.DataFrame(index=frameworks, columns=frameworks, dtype=float)

for fw_train in frameworks:
    X_train = get_layer(data[fw_train]["states"], TRANSFER_LAYER_POS)
    y_train = data[fw_train]["meta"]["label"].values
    sc, clf = train_probe(X_train, y_train)

    for fw_test in frameworks:
        X_test = get_layer(data[fw_test]["states"], TRANSFER_LAYER_POS)
        y_test = data[fw_test]["meta"]["label"].values
        auc = eval_probe(sc, clf, X_test, y_test)
        transfer_matrix.loc[fw_train, fw_test] = auc

print(f"Transfer matrix — Layer 32 (train → test):")
print(transfer_matrix.round(3).to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(
    transfer_matrix.astype(float), annot=True, fmt=".3f",
    vmin=0.5, vmax=1.0, cmap="RdYlGn", ax=ax,
    xticklabels=[f"Test:\n{f}" for f in frameworks],
    yticklabels=[f"Train: {f}" for f in frameworks],
)
ax.set_title("Harmful vs Benign AUC — Transfer matrix (Layer 32)")
plt.tight_layout()
plt.savefig(FIG_DIR / "transfer_matrix_layer32.png", dpi=150)
plt.show()

In [ ]:
# Transfer matrix for all layers
fig, axes = plt.subplots(1, len(LAYER_INDICES), figsize=(5 * len(LAYER_INDICES), 4), sharey=True)

for ax, li, lname in zip(axes, range(len(LAYER_INDICES)), LAYER_NAMES):
    mat = pd.DataFrame(index=frameworks, columns=frameworks, dtype=float)
    for fw_train in frameworks:
        X_train = get_layer(data[fw_train]["states"], li)
        y_train = data[fw_train]["meta"]["label"].values
        sc, clf = train_probe(X_train, y_train)
        for fw_test in frameworks:
            X_test = get_layer(data[fw_test]["states"], li)
            y_test = data[fw_test]["meta"]["label"].values
            mat.loc[fw_train, fw_test] = eval_probe(sc, clf, X_test, y_test)

    sns.heatmap(
        mat.astype(float), annot=True, fmt=".3f",
        vmin=0.5, vmax=1.0, cmap="RdYlGn", ax=ax,
        xticklabels=[f"Test: {f}" for f in frameworks],
        yticklabels=[f"Train: {f}" for f in frameworks] if li == 0 else False,
    )
    ax.set_title(lname)

fig.suptitle("Transfer matrix — all layers", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "transfer_matrix_all_layers.png", dpi=150, bbox_inches="tight")
plt.show()

## Trajectory comparison

Train a single LAT probe on Crescendo (harmful vs benign), then project
turns from ALL frameworks onto it turn by turn.

If trajectories look similar across frameworks, the Crescendo LAT direction captures
a universal harm encoding. If they diverge, each framework activates a different direction.

In [ ]:
# Train LAT on Crescendo
X_cresc = get_layer(data["crescendo"]["states"], TRANSFER_LAYER_POS)
y_cresc = data["crescendo"]["meta"]["label"].values
sc_lat, lat_vec = train_lat(X_cresc, y_cresc)

# Score all frameworks
fw_colors   = {"crescendo": "#e15759", "actorattack": "#4e79a7", "xteaming": "#59a14f"}
split_styles = {"harmful": "-", "benign": "--"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: harmful only, by framework
ax = axes[0]
for fw in frameworks:
    meta   = data[fw]["meta"]
    states = data[fw]["states"]
    mask   = meta["label"].values == 1
    if mask.sum() == 0:
        continue
    X_harm  = get_layer(states[mask], TRANSFER_LAYER_POS)
    scores  = sc_lat.transform(X_harm) @ lat_vec
    meta_h  = meta[mask].copy()
    meta_h["lat"] = scores
    traj = meta_h.groupby("turn_idx")["lat"].agg(["mean", "sem"])
    ax.plot(traj.index, traj["mean"], color=fw_colors[fw], label=fw, linewidth=2)
    ax.fill_between(traj.index, traj["mean"] - traj["sem"],
                    traj["mean"] + traj["sem"], color=fw_colors[fw], alpha=0.15)

ax.set_xlabel("Turn index")
ax.set_ylabel("LAT score (Crescendo harmful direction, Layer 32)")
ax.set_title("Harmful trajectories — Crescendo LAT probe")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Right: harmful vs benign per framework
ax = axes[1]
for fw in frameworks:
    meta   = data[fw]["meta"]
    states = data[fw]["states"]
    for split_val, split_name, ls in [(1, "harmful", "-"), (0, "benign", "--")]:
        mask = meta["label"].values == split_val
        if mask.sum() == 0:
            continue
        X_s    = get_layer(states[mask], TRANSFER_LAYER_POS)
        scores = sc_lat.transform(X_s) @ lat_vec
        meta_s = meta[mask].copy()
        meta_s["lat"] = scores
        traj   = meta_s.groupby("turn_idx")["lat"].agg(["mean", "sem"])
        ax.plot(traj.index, traj["mean"], color=fw_colors[fw],
                linestyle=ls, label=f"{fw} {split_name}", linewidth=1.5)

ax.set_xlabel("Turn index")
ax.set_ylabel("LAT score")
ax.set_title("Harmful (—) vs Benign (--) per framework")
ax.legend(fontsize=8)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(FIG_DIR / "transfer_trajectory.png", dpi=150)
plt.show()

## Outcome prediction transfer

Train probe on Crescendo jailbroken vs refusal, apply to ActorAttack/X-Teaming.
Tests whether the model's early-turn representation predicts eventual jailbreak
across frameworks.

In [ ]:
# Train outcome probe on Crescendo harmful (jailbroken vs refusal)
meta_c = data["crescendo"]["meta"]
mask_c = meta_c["final_verdict"].isin(["jailbroken", "refusal"]) & (meta_c["label"] == 1)
X_c    = get_layer(data["crescendo"]["states"][mask_c], TRANSFER_LAYER_POS)
y_c    = (meta_c[mask_c]["final_verdict"] == "jailbroken").astype(int).values

sc_out, clf_out = train_probe(X_c, y_c)

print("Outcome (jailbroken vs refusal) prediction — probe trained on Crescendo:")
print(f"  Crescendo (within, train set):  AUC = {eval_probe(sc_out, clf_out, X_c, y_c):.3f}  "
      f"(upper bound — trained on same data)")

for fw in ["actorattack", "xteaming"]:
    meta_t = data[fw]["meta"]
    mask_t = meta_t["final_verdict"].isin(["jailbroken", "refusal"]) & (meta_t["label"] == 1)
    if mask_t.sum() < 10:
        print(f"  {fw}: not enough jailbroken/refusal turns")
        continue
    X_t = get_layer(data[fw]["states"][mask_t], TRANSFER_LAYER_POS)
    y_t = (meta_t[mask_t]["final_verdict"] == "jailbroken").astype(int).values
    auc = eval_probe(sc_out, clf_out, X_t, y_t)
    print(f"  {fw} (transfer):              AUC = {auc:.3f}  "
          f"(jailbroken={y_t.sum()}, refusal={(y_t==0).sum()})")

# By turn — does transfer improve with more context?
fig, ax = plt.subplots(figsize=(8, 4))
for fw, color in fw_colors.items():
    meta_t = data[fw]["meta"]
    mask_fw = meta_t["final_verdict"].isin(["jailbroken", "refusal"]) & (meta_t["label"] == 1)
    rows = []
    for turn in sorted(meta_t["turn_idx"].unique()):
        mask_t = mask_fw & (meta_t["turn_idx"] == turn)
        if mask_t.sum() < 10:
            continue
        y_t = (meta_t[mask_t]["final_verdict"] == "jailbroken").astype(int).values
        if len(np.unique(y_t)) < 2:
            continue
        X_t = get_layer(data[fw]["states"][mask_t.values], TRANSFER_LAYER_POS)
        auc = eval_probe(sc_out, clf_out, X_t, y_t)
        rows.append({"turn": turn, "auc": auc})
    if rows:
        df_r = pd.DataFrame(rows)
        ax.plot(df_r["turn"], df_r["auc"], marker="o", color=color, label=fw)

ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("Turn index")
ax.set_ylabel("AUC")
ax.set_title("Outcome prediction by turn\n(probe trained on Crescendo, applied to all frameworks)")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig(FIG_DIR / "transfer_outcome_by_turn.png", dpi=150)
plt.show()

## Compliance depth effect across frameworks

For Crescendo, `n_context_turns` varies due to rollback.
For ActorAttack/X-Teaming, `n_context_turns = turn_idx - 1` (no rollback).

Project all harmful turns onto the Crescendo LAT direction and plot
mean score by context depth per framework. Tests whether compliance
accumulation shifts the representation similarly across frameworks.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: mean LAT score by n_context_turns per framework (harmful only)
ax = axes[0]
for fw, color in fw_colors.items():
    meta   = data[fw]["meta"]
    states = data[fw]["states"]
    mask   = meta["label"].values == 1
    X_h    = get_layer(states[mask], TRANSFER_LAYER_POS)
    scores = sc_lat.transform(X_h) @ lat_vec
    meta_h = meta[mask].copy()
    meta_h["lat"] = scores
    traj = meta_h.groupby("n_context_turns")["lat"].agg(["mean", "sem", "count"])
    traj = traj[traj["count"] >= 5]
    ax.plot(traj.index, traj["mean"], marker="o", color=color, label=fw, linewidth=2)
    ax.fill_between(traj.index, traj["mean"] - traj["sem"],
                    traj["mean"] + traj["sem"], color=color, alpha=0.15)

ax.set_xlabel("n_context_turns")
ax.set_ylabel("LAT score (Crescendo harmful direction)")
ax.set_title("Representation vs compliance depth\n(harmful turns only)")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Right: jailbroken rate by n_context_turns per framework (harmful only)
ax = axes[1]
for fw, color in fw_colors.items():
    meta = data[fw]["meta"]
    mask = meta["label"].values == 1
    meta_h = meta[mask].copy()
    meta_h["jailbroken"] = (meta_h["final_verdict"] == "jailbroken").astype(int)
    rate = meta_h.groupby("n_context_turns")["jailbroken"].agg(["mean", "count"])
    rate = rate[rate["count"] >= 10]
    ax.plot(rate.index, rate["mean"], marker="o", color=color, label=fw, linewidth=2)

ax.set_xlabel("n_context_turns")
ax.set_ylabel("Jailbroken rate")
ax.set_title("Jailbroken rate by compliance depth\n(harmful turns only)")
ax.legend()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(FIG_DIR / "transfer_compliance_depth.png", dpi=150)
plt.show()

## Summary table

In [ ]:
print("=" * 70)
print("CROSS-FRAMEWORK TRANSFER SUMMARY (Layer 32)")
print("=" * 70)

print("\nWithin-framework AUC (harmful vs benign):")
for fw in frameworks:
    auc = within_aucs.get((fw, "Layer 32"), float("nan"))
    print(f"  {fw:<15}: {auc:.3f}")

print("\nTransfer AUC (train Crescendo → test other):")
for fw in frameworks:
    auc = transfer_matrix.loc["crescendo", fw]
    print(f"  → {fw:<15}: {auc:.3f}")

print("\nTransfer efficiency (transfer / within-framework):")
for fw in ["actorattack", "xteaming"]:
    transfer = transfer_matrix.loc["crescendo", fw]
    within   = within_aucs.get((fw, "Layer 32"), float("nan"))
    print(f"  {fw:<15}: {transfer:.3f} / {within:.3f} = {transfer/within:.2%}")